# YOLO11 Corrosion Detection Training Pipeline
## Rebar & Steel Plate Corrosion Detection for Drone-based Bridge Inspection

This notebook:
1. Automatically downloads corrosion datasets from multiple sources
2. Merges and prepares datasets for training
3. Trains YOLO11n, YOLO11s, and YOLO11m models
4. Includes automatic resume capability for interrupted training
5. Saves results to Google Drive

**Best Datasets Identified:**
- `cawilcorrosionsegmentation/instance-corrosion`: Instance segmentation (1,009 images)
- `yatishs-workspace/corrosion-detection-z8iw3`: Instance segmentation (9.22k images)
- `rustwatch-4815v/corrosion-detection-ruyn1`: Object detection (1k images, multiple rust types)
- `university-of-tebessa/corrosion-eh3ms`: Steel corrosion classification (440 images)
- `corrosion-l4cqs/corrosion-detection-npdhd`: Large detection dataset (2.63k images)
- `thyssenkrupp/corrosion-whmsc`: Steel corrosion segmentation (1.76k images)
- `identification-pittin-corrosion/corrosion-segmentation-rsfbe`: Multi-class (421 images)


## Step 1: Mount Google Drive & Setup Environment

In [46]:
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
import os
WORKSPACE = '/content/corrosion_workspace'
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

# Drive paths for persistence
DRIVE_BASE = '/content/drive/MyDrive/YOLO_Corrosion_Training'
DRIVE_DATASETS = f'{DRIVE_BASE}/datasets'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_RESULTS = f'{DRIVE_BASE}/results'

for path in [DRIVE_BASE, DRIVE_DATASETS, DRIVE_MODELS, DRIVE_RESULTS]:
    os.makedirs(path, exist_ok=True)

print(f"Working in: {WORKSPACE}")
print(f"Drive base: {DRIVE_BASE}")
print(f"Drive datasets: {DRIVE_DATASETS}")
print(f"Drive models: {DRIVE_MODELS}")
print(f"Drive results: {DRIVE_RESULTS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/corrosion_workspace
Drive base: /content/drive/MyDrive/YOLO_Corrosion_Training
Drive datasets: /content/drive/MyDrive/YOLO_Corrosion_Training/datasets
Drive models: /content/drive/MyDrive/YOLO_Corrosion_Training/models
Drive results: /content/drive/MyDrive/YOLO_Corrosion_Training/results


## Step 2: Install Dependencies

In [47]:
# Install required packages
!pip install ultralytics roboflow kaggle -q

import cv2
import numpy as np
import yaml
import json
import shutil
import zipfile
from pathlib import Path
from glob import glob
import random
from collections import defaultdict
from tqdm import tqdm
import requests
from datetime import datetime
import pickle

print("All dependencies installed successfully!")

All dependencies installed successfully!


## Step 3: Resume State Management

This section handles automatic resume from interrupted training sessions.

In [48]:
class TrainingState:
    """Manages training state for resume capability"""
    
    def __init__(self, state_file=f'{DRIVE_BASE}/training_state.json'):
        self.state_file = state_file
        self.state = self.load_state()
    
    def load_state(self):
        if os.path.exists(self.state_file):
            with open(self.state_file, 'r') as f:
                return json.load(f)
        return {
            'datasets_downloaded': {},
            'datasets_merged': False,
            'models_trained': {},
            'last_epoch': {},
            'completed': False,
            'timestamp': None
        }
    
    def save_state(self):
        self.state['timestamp'] = datetime.now().isoformat()
        with open(self.state_file, 'w') as f:
            json.dump(self.state, f, indent=2)
    
    def mark_dataset_downloaded(self, dataset_name, version):
        self.state['datasets_downloaded'][dataset_name] = version
        self.save_state()
    
    def is_dataset_downloaded(self, dataset_name):
        return dataset_name in self.state['datasets_downloaded']
    
    def mark_datasets_merged(self):
        self.state['datasets_merged'] = True
        self.save_state()
    
    def mark_model_trained(self, model_name, epochs_completed, best_mAP50):
        self.state['models_trained'][model_name] = {
            'epochs_completed': epochs_completed,
            'best_mAP50': best_mAP50,
            'timestamp': datetime.now().isoformat()
        }
        self.save_state()
    
    def is_model_trained(self, model_name):
        return model_name in self.state['models_trained']
    
    def get_training_progress(self, model_name):
        return self.state['models_trained'].get(model_name, {})
    
    def print_status(self):
        print("\n=== Training State ===")
        print(f"Datasets Downloaded: {list(self.state['datasets_downloaded'].keys())}")
        print(f"Datasets Merged: {self.state['datasets_merged']}")
        print(f"Models Trained: {list(self.state['models_trained'].keys())}")
        if self.state['timestamp']:
            print(f"Last Update: {self.state['timestamp']}")
        print("====================\n")

# Initialize state manager
state = TrainingState()
state.print_status()


=== Training State ===
Datasets Downloaded: ['instance_corrosion', 'corrosion_detection_large', 'thyssenkrupp_steel', 'university_steel_corrosion', 'large_corrosion_detection', 'multi_type_corrosion', 'corrosion_types_4class', 'scaledge_corrosion', 'drima_corrosion']
Datasets Merged: True
Models Trained: []
Last Update: 2026-07-27T19:08:23.099622



## Step 4: Dataset Download Functions

Functions to download datasets from Roboflow and other sources.

In [49]:
from roboflow import Roboflow
import shutil

# Define datasets to download - prioritized list for corrosion detection
# Format: (workspace/project, version, dataset_name, task_type)
DATASETS_TO_DOWNLOAD = [
    # Primary datasets for rebar/steel corrosion
    ('cawilcorrosionsegmentation/instance-corrosion', 2, 'instance_corrosion', 'segmentation'),  # Best instance segmentation
    ('yatishs-workspace/corrosion-detection-z8iw3', 1, 'corrosion_detection_large', 'segmentation'),  # Large dataset
    ('rustwatch-4815v/corrosion-detection-ruyn1', 1, 'rust_types_detection', 'detection'),  # Multiple rust types
    ('thyssenkrupp/corrosion-whmsc', 1, 'thyssenkrupp_steel', 'segmentation'),  # Steel corrosion
    ('university-of-tebessa/corrosion-eh3ms', 1, 'university_steel_corrosion', 'detection'),  # Steel plate corrosion
    ('corrosion-l4cqs/corrosion-detection-npdhd', 4, 'large_corrosion_detection', 'detection'),  # Large general
    ('identification-pittin-corrosion/corrosion-segmentation-rsfbe', 1, 'multi_type_corrosion', 'segmentation'),  # Pitting, uniform, crack
    ('godson-0hjq0/corrosion-detection-0pt6o', 1, 'corrosion_types_4class', 'segmentation'),  # 4 corrosion types
    ('scaledge-ztihl/corrosion-detection-obirw', 1, 'scaledge_corrosion', 'detection'),  # Detection dataset
    ('drima/corrosion-zjnrh', 1, 'drima_corrosion', 'detection'),  # Detection
]

def download_from_roboflow(api_key, workspace_project, version, dataset_name, task_type, output_dir):
    """Download a dataset from Roboflow"""
    try:
        print(f"\nDownloading {dataset_name} from Roboflow...")
        
        rf = Roboflow(api_key=api_key)
        workspace, project = workspace_project.split('/')
        project_obj = rf.workspace(workspace).project(project)
        
        # Download in YOLOv8 format (works for YOLO11)
        dataset = project_obj.version(version).download("yolov8", location=output_dir)
        
        print(f"✓ Downloaded {dataset_name} to {output_dir}")
        return True
    except Exception as e:
        print(f"✗ Failed to download {dataset_name}: {str(e)}")
        return False

def download_all_datasets(roboflow_api_key, skip_downloaded=True):
    """Download all corrosion datasets"""
    datasets_dir = f'{WORKSPACE}/datasets'
    os.makedirs(datasets_dir, exist_ok=True)
    
    downloaded_paths = {}
    
    for workspace_project, version, dataset_name, task_type in DATASETS_TO_DOWNLOAD:
        output_dir = f'{datasets_dir}/{dataset_name}'
        
        # Check if already downloaded and in state
        if skip_downloaded and state.is_dataset_downloaded(dataset_name):
            print(f"Skipping {dataset_name} (already downloaded)")
            downloaded_paths[dataset_name] = output_dir
            continue
        
        # Check if directory exists
        if skip_downloaded and os.path.exists(output_dir):
            print(f"Skipping {dataset_name} (directory exists)")
            state.mark_dataset_downloaded(dataset_name, version)
            downloaded_paths[dataset_name] = output_dir
            continue
        
        # Download dataset
        success = download_from_roboflow(
            roboflow_api_key, 
            workspace_project, 
            version, 
            dataset_name, 
            task_type, 
            output_dir
        )
        
        if success:
            state.mark_dataset_downloaded(dataset_name, version)
            downloaded_paths[dataset_name] = output_dir
        
        # Copy to Drive for persistence
        drive_path = f'{DRIVE_DATASETS}/{dataset_name}'
        if os.path.exists(output_dir) and not os.path.exists(drive_path):
            shutil.copytree(output_dir, drive_path, dirs_exist_ok=True)
            print(f"  Copied to Drive: {drive_path}")
    
    return downloaded_paths

# You'll need to add your Roboflow API key here
ROBOFLOW_API_KEY = None  # Will prompt user

print("Dataset download functions ready!")
print(f"\nDatasets to download: {len(DATASETS_TO_DOWNLOAD)}")
for wp, ver, name, task in DATASETS_TO_DOWNLOAD:
    print(f"  - {name} ({task}): {wp}")

Dataset download functions ready!

Datasets to download: 10
  - instance_corrosion (segmentation): cawilcorrosionsegmentation/instance-corrosion
  - corrosion_detection_large (segmentation): yatishs-workspace/corrosion-detection-z8iw3
  - rust_types_detection (detection): rustwatch-4815v/corrosion-detection-ruyn1
  - thyssenkrupp_steel (segmentation): thyssenkrupp/corrosion-whmsc
  - university_steel_corrosion (detection): university-of-tebessa/corrosion-eh3ms
  - large_corrosion_detection (detection): corrosion-l4cqs/corrosion-detection-npdhd
  - multi_type_corrosion (segmentation): identification-pittin-corrosion/corrosion-segmentation-rsfbe
  - corrosion_types_4class (segmentation): godson-0hjq0/corrosion-detection-0pt6o
  - scaledge_corrosion (detection): scaledge-ztihl/corrosion-detection-obirw
  - drima_corrosion (detection): drima/corrosion-zjnrh


## Step 5: Set Roboflow API Key and Download Datasets

In [50]:
# Set your Roboflow API key
# Get your API key from: https://app.roboflow.com/api-key
ROBOFLOW_API_KEY = "5Zmgm6NCmNWapNJMTOtk"

if ROBOFLOW_API_KEY:
    # Download all datasets
    downloaded_datasets = download_all_datasets(ROBOFLOW_API_KEY, skip_downloaded=True)
    print(f"\n✓ Downloaded {len(downloaded_datasets)} datasets")
else:
    print("No API key provided. Using cached datasets if available.")
    
    # Try to restore from Drive
    downloaded_datasets = {}
    datasets_dir = f'{WORKSPACE}/datasets'
    os.makedirs(datasets_dir, exist_ok=True)
    
    for _, _, name, _ in DATASETS_TO_DOWNLOAD:
        drive_path = f'{DRIVE_DATASETS}/{name}'
        local_path = f'{datasets_dir}/{name}'
        
        if os.path.exists(drive_path) and not os.path.exists(local_path):
            shutil.copytree(drive_path, local_path, dirs_exist_ok=True)
            print(f"Restored from Drive: {name}")
            downloaded_datasets[name] = local_path
        elif os.path.exists(local_path):
            print(f"Using local: {name}")
            downloaded_datasets[name] = local_path
    
    print(f"\nUsing {len(downloaded_datasets)} cached datasets")

Skipping instance_corrosion (already downloaded)
Skipping corrosion_detection_large (already downloaded)

loading Roboflow workspace...
loading Roboflow project...
✗ Failed to download rust_types_detection: Version number 1 is not found.
Skipping thyssenkrupp_steel (already downloaded)
Skipping university_steel_corrosion (already downloaded)
Skipping large_corrosion_detection (already downloaded)
Skipping multi_type_corrosion (already downloaded)
Skipping corrosion_types_4class (already downloaded)
Skipping scaledge_corrosion (already downloaded)
Skipping drima_corrosion (already downloaded)

✓ Downloaded 9 datasets


## Step 6: Dataset Analysis and Merging

Analyze downloaded datasets and merge them into a unified training set.

In [51]:
def analyze_dataset(dataset_path):
    """Analyze a YOLO format dataset"""
    info = {
        'path': dataset_path,
        'name': os.path.basename(dataset_path),
        'train_images': 0,
        'val_images': 0,
        'test_images': 0,
        'classes': [],
        'total_annotations': 0,
        'task': 'unknown'
    }
    
    # Read data.yaml if exists
    yaml_path = os.path.join(dataset_path, 'data.yaml')
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r') as f:
            data = yaml.safe_load(f)
            info['classes'] = data.get('names', [])
            info['nc'] = data.get('nc', len(info['classes']))
    
    # Count images
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(dataset_path, split, 'images')
        if os.path.exists(img_dir):
            count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
            if split == 'train':
                info['train_images'] = count
            elif split == 'valid':
                info['val_images'] = count
            else:
                info['test_images'] = count
        
        # Count labels
        label_dir = os.path.join(dataset_path, split, 'labels')
        if os.path.exists(label_dir):
            for lbl_file in os.listdir(label_dir):
                if lbl_file.endswith('.txt'):
                    lbl_path = os.path.join(label_dir, lbl_file)
                    with open(lbl_path, 'r') as f:
                        info['total_annotations'] += len(f.readlines())
    
    # Detect task type from label files
    sample_label = None
    for split in ['train', 'valid', 'test']:
        label_dir = os.path.join(dataset_path, split, 'labels')
        if os.path.exists(label_dir):
            label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
            if label_files:
                sample_label = os.path.join(label_dir, label_files[0])
                break
    
    if sample_label:
        with open(sample_label, 'r') as f:
            first_line = f.readline().strip()
            if first_line:
                parts = first_line.split()
                if len(parts) == 5:
                    info['task'] = 'detection'  # class, x_center, y_center, width, height
                elif len(parts) > 5 and len(parts) % 2 == 1:
                    info['task'] = 'segmentation'  # class, x1, y1, x2, y2, ...
    
    return info

def standardize_dataset(dataset_path, unified_classes, output_path):
    """Convert dataset to unified class names and save to merged dataset"""
    
    # Read original data.yaml
    yaml_path = os.path.join(dataset_path, 'data.yaml')
    if not os.path.exists(yaml_path):
        print(f"Warning: No data.yaml found in {dataset_path}")
        return
    
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    original_classes = data.get('names', [])
    
    # Create class mapping
    class_mapping = {}
    for i, orig_class in enumerate(original_classes):
        orig_lower = orig_class.lower()
        
        # Map to unified classes
        if any(x in orig_lower for x in ['corrosion', 'rust', 'corroded']):
            if any(x in orig_lower for x in ['severe', 'poor', 'high']) or '4' in orig_lower:
                class_mapping[i] = 'severe_corrosion'
            elif any(x in orig_lower for x in ['moderate', 'fair', 'medium']) or '3' in orig_lower:
                class_mapping[i] = 'moderate_corrosion'
            elif any(x in orig_lower for x in ['mild', 'low', 'light']) or '2' in orig_lower:
                class_mapping[i] = 'mild_corrosion'
            elif 'pitting' in orig_lower:
                class_mapping[i] = 'pitting_corrosion'
            elif 'uniform' in orig_lower:
                class_mapping[i] = 'uniform_corrosion'
            elif 'crevice' in orig_lower:
                class_mapping[i] = 'crevice_corrosion'
            else:
                class_mapping[i] = 'corrosion'
        elif any(x in orig_lower for x in ['clean', 'no-corroded', 'non-corroded']):
            class_mapping[i] = 'no_corrosion'
        elif any(x in orig_lower for x in ['rebar', 'bar']):
            class_mapping[i] = 'rebar_corrosion'
        elif 'crack' in orig_lower:
            class_mapping[i] = 'crack'  # Keep cracks for multi-task learning
        else:
            class_mapping[i] = 'corrosion'
    
    # Process each split
    for split in ['train', 'valid', 'test']:
        src_img_dir = os.path.join(dataset_path, split, 'images')
        src_lbl_dir = os.path.join(dataset_path, split, 'labels')
        
        if not os.path.exists(src_img_dir):
            continue
        
        dst_img_dir = os.path.join(output_path, split, 'images')
        dst_lbl_dir = os.path.join(output_path, split, 'labels')
        os.makedirs(dst_img_dir, exist_ok=True)
        os.makedirs(dst_lbl_dir, exist_ok=True)
        
        for img_file in tqdm(os.listdir(src_img_dir), desc=f"Processing {split}", leave=False):
            if not img_file.endswith(('.jpg', '.jpeg', '.png')):
                continue
            
            # Copy image
            src_img = os.path.join(src_img_dir, img_file)
            dataset_name = os.path.basename(dataset_path)
            new_img_name = f"{dataset_name}_{img_file}"
            dst_img = os.path.join(dst_img_dir, new_img_name)
            shutil.copy2(src_img, dst_img)
            
            # Process labels
            lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
            src_lbl = os.path.join(src_lbl_dir, lbl_file)
            dst_lbl = os.path.join(dst_lbl_dir, new_img_name.rsplit('.', 1)[0] + '.txt')
            
            if os.path.exists(src_lbl):
                with open(src_lbl, 'r') as f_in, open(dst_lbl, 'w') as f_out:
                    for line in f_in:
                        parts = line.strip().split()
                        if not parts:
                            continue
                        orig_class = int(parts[0])
                        
                        if orig_class in class_mapping:
                            unified_class = class_mapping[orig_class]
                            if unified_class in unified_classes:
                                new_class_id = unified_classes.index(unified_class)
                                f_out.write(f"{new_class_id} {' '.join(parts[1:])}\n")

# Analyze all downloaded datasets
print("\n=== Analyzing Downloaded Datasets ===\n")
dataset_infos = []
for name, path in downloaded_datasets.items():
    if os.path.exists(path):
        info = analyze_dataset(path)
        dataset_infos.append(info)
        print(f"\n{name}:")
        print(f"  Task: {info['task']}")
        print(f"  Classes: {info['classes']}")
        print(f"  Train: {info['train_images']}, Val: {info['val_images']}, Test: {info['test_images']}")
        print(f"  Total annotations: {info['total_annotations']}")

print(f"\nTotal datasets analyzed: {len(dataset_infos)}")


=== Analyzing Downloaded Datasets ===


instance_corrosion:
  Task: segmentation
  Classes: ['Corrosion']
  Train: 2106, Val: 202, Test: 98
  Total annotations: 11151

corrosion_detection_large:
  Task: segmentation
  Classes: ['1', '10', '11', '12', '13', '14', '14 0 0 0 1 1 1 1 0 0 0', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
  Train: 17198, Val: 1844, Test: 922
  Total annotations: 76560

thyssenkrupp_steel:
  Task: segmentation
  Classes: ['2_Fair_Steel_Corrosion', '3_Poor_Steel_Corrosion', '4_Severe_Steel_Corrosion']
  Train: 1585, Val: 0, Test: 176
  Total annotations: 16327

university_steel_corrosion:
  Task: segmentation
  Classes: ['2_Fair_Steel_Corrosion', '3_Poor_Steel_Corrosion', '4_Severe_Steel_Corrosion']
  Train: 352, Val: 44, Test: 44
  Total annotations: 4107

large_corrosion_detection:
  Task: detection
  Classes: ['corrosion']
  Train: 1846, Val: 390, Test: 389
  Total annotations: 5337

multi_type_corrosion:
  Task: segmentation
  Clas

## Step 7: Create Unified Dataset

In [52]:
# Check if merged dataset already exists
MERGED_DATASET_PATH = f'{WORKSPACE}/merged_corrosion_dataset'

if state.state['datasets_merged'] and os.path.exists(MERGED_DATASET_PATH):
    print("Using existing merged dataset from previous run")
else:
    # Define unified class names for corrosion detection
    # Focusing on severity levels and types relevant to bridge inspection
    UNIFIED_CLASSES = [
        'corrosion',          # General corrosion
        'mild_corrosion',     # Early stage
        'moderate_corrosion', # Developing
        'severe_corrosion',   # Advanced damage
        'pitting_corrosion',  # Localized pitting
        'uniform_corrosion',  # Even surface corrosion
        'rebar_corrosion',    # Rebar-specific
        'steel_corrosion',    # Steel plate-specific
        'crack',              # Structural cracks (for multi-task)
    ]
    
    print(f"Creating unified dataset with {len(UNIFIED_CLASSES)} classes...")
    print(f"Classes: {UNIFIED_CLASSES}")
    
    # Create merged dataset
    os.makedirs(MERGED_DATASET_PATH, exist_ok=True)
    
    # Process each downloaded dataset
    for name, path in tqdm(downloaded_datasets.items(), desc="Merging datasets"):
        if os.path.exists(path):
            standardize_dataset(path, UNIFIED_CLASSES, MERGED_DATASET_PATH)
    
    # Count merged dataset
    total_train = len([f for f in os.listdir(f'{MERGED_DATASET_PATH}/train/images') if f.endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(f'{MERGED_DATASET_PATH}/train/images') else 0
    total_val = len([f for f in os.listdir(f'{MERGED_DATASET_PATH}/valid/images') if f.endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(f'{MERGED_DATASET_PATH}/valid/images') else 0
    total_test = len([f for f in os.listdir(f'{MERGED_DATASET_PATH}/test/images') if f.endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(f'{MERGED_DATASET_PATH}/test/images') else 0
    
    print(f"\n=== Merged Dataset Statistics ===")
    print(f"Train images: {total_train}")
    print(f"Val images: {total_val}")
    print(f"Test images: {total_test}")
    print(f"Total images: {total_train + total_val + total_test}")
    
    # Create data.yaml for merged dataset
    data_yaml = {
        'path': MERGED_DATASET_PATH,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(UNIFIED_CLASSES),
        'names': UNIFIED_CLASSES
    }
    
    yaml_path = os.path.join(MERGED_DATASET_PATH, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    
    print(f"\nCreated data.yaml at {yaml_path}")
    
    # Mark as merged
    state.mark_datasets_merged()
    
    # Copy to Drive
    drive_merged = f'{DRIVE_DATASETS}/merged_corrosion_dataset'
    if os.path.exists(drive_merged):
        shutil.rmtree(drive_merged)
    shutil.copytree(MERGED_DATASET_PATH, drive_merged)
    print(f"\nCopied merged dataset to Drive: {drive_merged}")

# Verify merged dataset
yaml_path = os.path.join(MERGED_DATASET_PATH, 'data.yaml')
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)
    print(f"\nDataset YAML contents:")
    print(data)

Using existing merged dataset from previous run

Dataset YAML contents:
{'names': ['corrosion', 'mild_corrosion', 'moderate_corrosion', 'severe_corrosion', 'pitting_corrosion', 'uniform_corrosion', 'rebar_corrosion', 'steel_corrosion', 'crack'], 'nc': 9, 'path': '/content/corrosion_workspace/merged_corrosion_dataset', 'test': 'test/images', 'train': 'train/images', 'val': 'valid/images'}


## Step 8: Data Augmentation Setup

Configure advanced augmentation for corrosion detection.

In [53]:
# We'll use YOLO's built-in augmentation, but let's also add custom augmentations if needed
from ultralytics import YOLO

# Define augmentation configuration
AUGMENTATION_CONFIG = {
    # Geometric augmentations
    'degrees': 15.0,      # Rotation (+/- degrees)
    'translate': 0.2,   # Translation (+/- fraction)
    'scale': 0.5,       # Scale (+/- gain)
    'shear': 5.0,       # Shear (+/- degrees)
    'perspective': 0.0, # Perspective (+/- fraction)
    'flipud': 0.5,      # Vertical flip probability
    'fliplr': 0.5,      # Horizontal flip probability
    
    # Color augmentations
    'hsv_h': 0.015,     # HSV-Hue augmentation
    'hsv_s': 0.7,       # HSV-Saturation augmentation
    'hsv_v': 0.4,       # HSV-Value augmentation
    'bgr': 0.0,         # BGR channel order probability
    
    # Mosaic and MixUp
    'mosaic': 1.0,      # Mosaic augmentation probability
    'mixup': 0.2,       # MixUp augmentation probability
    'copy_paste': 0.1,  # Copy-paste augmentation probability
    'erasing': 0.4,     # Random erasing probability
    'auto_augment': 'randaugment',  # Auto augmentation policy
    
    # Other
    'close_mosaic': 10, # Disable mosaic for last N epochs
}

print("Augmentation configuration ready!")
print(f"\nKey augmentations:")
for key, value in AUGMENTATION_CONFIG.items():
    print(f"  {key}: {value}")

Augmentation configuration ready!

Key augmentations:
  degrees: 15.0
  translate: 0.2
  scale: 0.5
  shear: 5.0
  perspective: 0.0
  flipud: 0.5
  fliplr: 0.5
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  bgr: 0.0
  mosaic: 1.0
  mixup: 0.2
  copy_paste: 0.1
  erasing: 0.4
  auto_augment: randaugment
  close_mosaic: 10


## Step 9: Training Configuration and Execution

Train YOLO11n, YOLO11s, and YOLO11m models with resume capability.

In [54]:
# Training configuration
TRAINING_CONFIG = {
    'data': os.path.join(MERGED_DATASET_PATH, 'data.yaml'),
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,
    'device': 0,
    'patience': 20,
    'save_period': 5,   # Save checkpoint every N epochs
    'exist_ok': True,
    'pretrained': True,
    'optimizer': 'auto',  # Auto-select (AdamW for segmentation)
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'overlap_mask': True,
    'mask_ratio': 4,
    'dropout': 0.0,
    'val': True,
    'plots': True,
    'verbose': True,
    'seed': 42,
    'deterministic': True,
    'single_cls': False,
    'rect': False,
    'cos_lr': False,
    'fraction': 1.0,
    'freeze': None,
}

# Add augmentation config
TRAINING_CONFIG.update(AUGMENTATION_CONFIG)

# Model variants to train - ONLY YOLO11m
MODELS = {
    'yolo11m': {
        'weights': 'yolo11m.pt',
        'name': 'yolo11m_corrosion',
        'epochs': 100,
        'batch': 8,
    },
}

print("Training configuration ready!")
print(f"\nModels to train: {list(MODELS.keys())}")
print(f"\nTraining config:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

Training configuration ready!

Models to train: ['yolo11n', 'yolo11s', 'yolo11m']

Training config:
  data: /content/corrosion_workspace/merged_corrosion_dataset/data.yaml
  epochs: 100
  imgsz: 640
  batch: 16
  device: 0
  patience: 20
  save_period: 5
  exist_ok: True
  pretrained: True
  optimizer: auto
  lr0: 0.01
  lrf: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3.0
  warmup_momentum: 0.8
  warmup_bias_lr: 0.1
  box: 7.5
  cls: 0.5
  dfl: 1.5
  overlap_mask: True
  mask_ratio: 4
  dropout: 0.0
  val: True
  plots: True
  verbose: True
  seed: 42
  deterministic: True
  single_cls: False
  rect: False
  cos_lr: False
  fraction: 1.0
  freeze: None
  degrees: 15.0
  translate: 0.2
  scale: 0.5
  shear: 5.0
  perspective: 0.0
  flipud: 0.5
  fliplr: 0.5
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  bgr: 0.0
  mosaic: 1.0
  mixup: 0.2
  copy_paste: 0.1
  erasing: 0.4
  auto_augment: randaugment
  close_mosaic: 10


## Step 10: Training Function with Resume Capability

In [55]:
def train_model(model_key, model_config, resume_from=None):
    """Train a single model with resume capability"""
    
    print(f"\n{'='*50}")
    print(f"Training {model_key}")
    print(f"{'='*50}")
    
    # Check if already trained
    if state.is_model_trained(model_key) and resume_from is None:
        progress = state.get_training_progress(model_key)
        print(f"Model {model_key} already trained: {progress}")
        return
    
    # Setup paths
    project_dir = f'{WORKSPACE}/runs'
    name = model_config['name']
    
    # Check for existing checkpoint
    checkpoint_dir = f'{project_dir}/{name}'
    last_pt = f'{checkpoint_dir}/weights/last.pt'
    
    if resume_from is None and os.path.exists(last_pt):
        print(f"Found existing checkpoint: {last_pt}")
        resume_from = last_pt
    
    # Load model
    if resume_from and os.path.exists(resume_from):
        print(f"Resuming from: {resume_from}")
        model = YOLO(resume_from)
        resume_flag = True
    else:
        print(f"Loading pretrained weights: {model_config['weights']}")
        model = YOLO(model_config['weights'])
        resume_flag = False
    
    # Prepare training arguments
    train_args = TRAINING_CONFIG.copy()
    train_args['epochs'] = model_config['epochs']
    train_args['batch'] = model_config['batch']
    train_args['project'] = project_dir
    train_args['name'] = name
    train_args['resume'] = resume_flag
    
    # Train
    try:
        results = model.train(**train_args)
        
        # Get best mAP
        best_mAP50 = results.results_dict.get('metrics/mAP50(M)', 0)
        
        # Mark as trained
        state.mark_model_trained(model_key, model_config['epochs'], best_mAP50)
        
        print(f"\n✓ {model_key} training completed!")
        print(f"  Best mAP50: {best_mAP50:.4f}")
        
        # Copy to Drive
        drive_model_dir = f'{DRIVE_MODELS}/{model_key}'
        os.makedirs(drive_model_dir, exist_ok=True)
        
        # Copy best weights
        best_pt = f'{checkpoint_dir}/weights/best.pt'
        if os.path.exists(best_pt):
            shutil.copy2(best_pt, f'{drive_model_dir}/best.pt')
            print(f"  Saved best weights to Drive")
        
        # Copy results
        results_files = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']
        for rf in results_files:
            src = f'{checkpoint_dir}/{rf}'
            if os.path.exists(src):
                shutil.copy2(src, drive_model_dir)
        
        print(f"  Saved results to Drive")
        
        return results
        
    except Exception as e:
        print(f"\n✗ {model_key} training failed: {str(e)}")
        print(f"  You can resume by running this cell again!")
        raise

print("Training function ready!")

Training function ready!


## Step 11: Execute Training

Train all three YOLO11 models. This will take several hours.

**Note:** If training is interrupted, simply re-run this cell to automatically resume!

In [ ]:
# Train all models
for model_key, model_config in MODELS.items():
    try:
        train_model(model_key, model_config)
    except Exception as e:
        print(f"Failed to train {model_key}: {e}")
        print("You can retry by re-running this cell. It will resume automatically.")
        break

print("\n=== Training Summary ===")
state.print_status()


Training yolo11n
Loading pretrained weights: yolo11n.pt
Ultralytics 8.4.107 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/corrosion_workspace/merged_corrosion_dataset/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=y

## Step 12: Model Evaluation and Comparison

In [ ]:
# Evaluate all trained models
print("\n=== Model Evaluation ===\n")

evaluation_results = {}

for model_key, model_config in MODELS.items():
    model_path = f'{WORKSPACE}/runs/{model_config["name"]}/weights/best.pt'
    
    if not os.path.exists(model_path):
        print(f"Skipping {model_key} (not trained yet)")
        continue
    
    print(f"\nEvaluating {model_key}...")
    
    try:
        model = YOLO(model_path)
        
        # Run validation
        results = model.val(
            data=os.path.join(MERGED_DATASET_PATH, 'data.yaml'),
            imgsz=640,
            batch=16,
            device=0,
            split='test' if os.path.exists(f'{MERGED_DATASET_PATH}/test/images') else 'val',
            save_json=True,
            save_hybrid=True,
        )
        
        # Extract metrics
        metrics = {
            'mAP50': results.results_dict.get('metrics/mAP50(M)', 0),
            'mAP50-95': results.results_dict.get('metrics/mAP50-95(M)', 0),
            'Precision': results.results_dict.get('metrics/precision(M)', 0),
            'Recall': results.results_dict.get('metrics/recall(M)', 0),
        }
        
        evaluation_results[model_key] = metrics
        
        print(f"  mAP50: {metrics['mAP50']:.4f}")
        print(f"  mAP50-95: {metrics['mAP50-95']:.4f}")
        print(f"  Precision: {metrics['Precision']:.4f}")
        print(f"  Recall: {metrics['Recall']:.4f}")
        
    except Exception as e:
        print(f"  Error evaluating {model_key}: {e}")

# Print comparison table
if evaluation_results:
    print("\n=== Model Comparison ===\n")
    print(f"{'Model':<15} {'mAP50':<10} {'mAP50-95':<10} {'Precision':<10} {'Recall':<10}")
    print("-" * 60)
    for model_key, metrics in evaluation_results.items():
        print(f"{model_key:<15} {metrics['mAP50']:.4f}     {metrics['mAP50-95']:.4f}     {metrics['Precision']:.4f}     {metrics['Recall']:.4f}")
    
    # Find best model
    best_model = max(evaluation_results.items(), key=lambda x: x[1]['mAP50'])
    print(f"\nBest model: {best_model[0]} with mAP50 = {best_model[1]['mAP50']:.4f}")


=== Model Evaluation ===

Skipping yolo11n-seg (not trained yet)
Skipping yolo11s-seg (not trained yet)
Skipping yolo11m-seg (not trained yet)


## Step 13: Export Models for Deployment

Export models to different formats for drone deployment.

In [ ]:
# Export models to various formats
EXPORT_FORMATS = ['onnx', 'torchscript', 'tflite', 'edgetpu', 'openvino']

for model_key, model_config in MODELS.items():
    model_path = f'{WORKSPACE}/runs/{model_config["name"]}/weights/best.pt'
    
    if not os.path.exists(model_path):
        continue
    
    print(f"\nExporting {model_key}...")
    
    try:
        model = YOLO(model_path)
        
        # Export to ONNX (recommended for most deployments)
        print("  Exporting to ONNX...")
        onnx_path = model.export(format='onnx', imgsz=640, dynamic=True)
        print(f"    ✓ ONNX: {onnx_path}")
        
        # Export to TorchScript
        print("  Exporting to TorchScript...")
        torchscript_path = model.export(format='torchscript', imgsz=640)
        print(f"    ✓ TorchScript: {torchscript_path}")
        
        # For edge devices: TFLite (optional, takes longer)
        # print("  Exporting to TFLite...")
        # tflite_path = model.export(format='tflite', imgsz=640)
        # print(f"    ✓ TFLite: {tflite_path}")
        
        # Copy exports to Drive
        export_dir = f'{WORKSPACE}/runs/{model_config["name"]}'
        drive_export = f'{DRIVE_MODELS}/{model_key}/exports'
        os.makedirs(drive_export, exist_ok=True)
        
        for export_file in ['*.onnx', '*.torchscript']:
            for f in glob(f'{export_dir}/{export_file}'):
                shutil.copy2(f, drive_export)
                print(f"    Copied to Drive: {os.path.basename(f)}")
        
    except Exception as e:
        print(f"  Error exporting {model_key}: {e}")

print("\n✓ Export completed!")


✓ Export completed!


## Step 14: Final Backup to Google Drive

In [ ]:
# Create final backup
print("Creating final backup...\n")

# Backup training state
shutil.copy2(f'{DRIVE_BASE}/training_state.json', f'{DRIVE_RESULTS}/training_state_backup.json')
print("✓ Backed up training state")

# Create summary report
summary = {
    'timestamp': datetime.now().isoformat(),
    'datasets': list(downloaded_datasets.keys()),
    'training_state': state.state,
    'evaluation_results': evaluation_results,
    'model_configs': MODELS
}

summary_path = f'{DRIVE_RESULTS}/training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved training summary to {summary_path}")

# List all saved files
print("\n=== Saved to Google Drive ===")
for root, dirs, files in os.walk(DRIVE_BASE):
    level = root.replace(DRIVE_BASE, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{sub_indent}{file}')

print("\n=== Training Complete! ===")
print(f"All models and results saved to: {DRIVE_BASE}")
print("\nTo resume training in a new session:")
print("  1. Mount Google Drive")
print("  2. Run Step 3 (Training State Management)")
print("  3. Run Step 11 (Execute Training) - it will automatically resume!")

Creating final backup...

✓ Backed up training state
✓ Saved training summary to /content/drive/MyDrive/YOLO_Corrosion_Training/results/training_summary.json

=== Saved to Google Drive ===
YOLO_Corrosion_Training/
  training_state.json
  datasets/
    instance_corrosion/
      README.dataset.txt
      README.roboflow.txt
      data.yaml
      train/
        images/
          Screenshot-2023-08-17-015718_png.rf.43b83f3da9456b57ca3fa1e8705a0aac.jpg
          Captura-de-Tela-2023-01-25-a-s-18-31-12_png_jpg.rf.5f11d085f592f52941c2035562ac20f9.jpg
          000155_jpg.rf.3583d9282160945e10b03de25da8f9d0.jpg
          CM9_jpg.rf.6f0ee0cd76f8acc2b7c6b1a9397923b3.jpg
          000435_jpg.rf.605dcd619b23f0f07c12d79f4d30a1cc.jpg
          Screenshot-2023-08-17-020336_png.rf.51aefa8a3a22c1639f240637c5192a92.jpg
          Captura-de-Tela-2023-01-25-a-s-18-22-58_png_jpg.rf.59e9db66c3c24d712029b3408d5b27d8.jpg
          Screenshot-2023-08-17-003408_png.rf.55c6fcdde87b953252efc34afc7f7270.jpg
       

## Step 15: Inference Demo

Test the trained model on sample images.

In [ ]:
# Inference demo
import matplotlib.pyplot as plt
from PIL import Image

def run_inference(model_path, image_path, conf=0.25, iou=0.45):
    """Run inference on a single image"""
    model = YOLO(model_path)
    results = model(image_path, conf=conf, iou=iou)
    
    # Plot results
    for r in results:
        im_array = r.plot()
        im = Image.fromarray(im_array[..., ::-1])
        return im
    
    return None

# Test on a validation image
val_img_dir = f'{MERGED_DATASET_PATH}/valid/images'
if os.path.exists(val_img_dir):
    val_images = [f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))][:5]
    
    if val_images:
        # Use best model
        best_model_key = max(evaluation_results.items(), key=lambda x: x[1]['mAP50'])[0] if evaluation_results else 'yolo11s-seg'
        best_model_path = f'{WORKSPACE}/runs/{MODELS[best_model_key]["name"]}/weights/best.pt'
        
        if os.path.exists(best_model_path):
            print(f"Running inference with {best_model_key}...\n")
            
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.flatten()
            
            for i, img_file in enumerate(val_images[:6]):
                img_path = os.path.join(val_img_dir, img_file)
                result_img = run_inference(best_model_path, img_path)
                
                if result_img:
                    axes[i].imshow(result_img)
                    axes[i].set_title(img_file[:30])
                    axes[i].axis('off')
            
            plt.tight_layout()
            plt.savefig(f'{DRIVE_RESULTS}/inference_demo.png', dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print(f"Model not found: {best_model_path}")
    else:
        print("No validation images found")
else:
    print(f"Validation directory not found: {val_img_dir}")

Model not found: /content/corrosion_workspace/runs/yolo11s_corrosion/weights/best.pt


## Troubleshooting and Resume Guide

### If training was interrupted:
1. Simply re-run the notebook from Step 1
2. The notebook will automatically detect and resume from the last checkpoint
3. State is saved in Google Drive, so it persists across sessions

### To reset and start fresh:
```python
# Uncomment and run to reset
# import shutil
# shutil.rmtree(DRIVE_BASE)
# state = TrainingState()
# print("Reset complete!")
```

### To resume specific model:
```python
# Resume YOLO11n
# train_model('yolo11n-seg', MODELS['yolo11n-seg'])
```

### To download only specific datasets:
Comment out unwanted datasets in `DATASETS_TO_DOWNLOAD` list in Step 4.